# Week 5 additional practical — Working with BAM files in Python

Open, query, inspect, and write a real BAM using **pysam**. You will build a tiny
synthetic dataset locally, then connect its alignment records to the pileups
used in the Week 5 lecture. No real genome or personal sequencing data are used.

This is **additional practice**, separate from the original Week 5 notebook.
Allow about 75–90 minutes: setup 5–10, headers/regions 15, flags/CIGAR 20,
pileup/coverage 20–25, writing/interpretation 15–20. It does not automatically
add to the original required homework; your instructor will specify its use.
The optional index experiment takes additional time.

Cells run as supplied. Predict, make the requested edits, and write explanations.
Running all cells unchanged does not complete the tasks.


## 0. Install once, then create your own tiny dataset

[Open this notebook in Colab](https://colab.research.google.com/github/vo-ya/etf_bioinformatics/blob/main/practicals/week-05/bam_exercise.ipynb)

Save a copy in Drive, or download this file and upload it to Colab. Use Python 3
and a CPU. This notebook needs **pysam**; the next cell installs version 0.24.1
only if the package is missing. Installation requires internet. It prints the
installed version; record it when sharing results. No external samtools executable
is required: pysam exposes the sorting/indexing commands used here.

After installation, all data generation and analysis are local. The supplied
builder creates FASTA, BAM, and index files in a fresh temporary directory and
prints its path. It does not overwrite your existing files. A runtime reset can
remove temporary files: rerun setup and then the following cells in order.
If installation is unavailable, use a prepared environment with pysam installed.


In [ ]:
import sys
import importlib.util
import subprocess
if sys.version_info < (3, 9):
    raise RuntimeError('Use Python 3.9 or later')
if importlib.util.find_spec('pysam') is None:
    # One package download, only when pysam is missing. Uses this notebook's Python.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pysam==0.24.1'])
import pysam
print('pysam:', pysam.__version__)
print('BAM setup ready. CPU only; no genome download.')


In [ ]:
#@title Load the supplied BAM builder and counting helpers
"""Build a tiny real BAM for teaching; this source is embedded in each notebook."""
from pathlib import Path
from collections import Counter
import tempfile
import pysam


def create_bam_example():
    """Use a fresh temporary directory so no existing student file is replaced."""
    work_dir = Path(tempfile.mkdtemp(prefix='week05_bam_'))
    letters = list('ACGT' * 50)
    letters[98:103] = 'TCAAT'
    reference = ''.join(letters)
    fasta_path = work_dir / 'toy.fa'
    fasta_path.write_text('>toy\n' + reference + '\n')
    pysam.faidx(str(fasta_path))
    header = {'HD': {'VN': '1.6', 'SO': 'unsorted'},
              'SQ': [{'SN': 'toy', 'LN': len(reference)}],
              'RG': [{'ID': 'class', 'SM': 'synthetic'}]}
    records = []

    def record(name, start, sequence, cigar, flag=0, mq=60, quality=35):
        read = pysam.AlignedSegment()
        read.query_name = name
        read.query_sequence = sequence  # Already in alignment/reference orientation.
        read.flag = flag
        read.reference_id = 0 if start is not None else -1
        read.reference_start = start if start is not None else -1
        read.mapping_quality = mq
        read.cigarstring = cigar
        read.query_qualities = pysam.qualitystring_to_array(chr(quality + 33) * len(sequence))
        read.set_tag('RG', 'class')
        records.append(read)
        return read

    for base in ('A', 'G'):
        for i in range(6):
            record(f'{base}_{i+1:02}', 98, 'TC' + base + 'AT', '5M', flag=16 if i % 2 else 0)
    for i in range(3):
        record(f'low_base_G_{i+1}', 98, 'TCGAT', '5M', quality=10)
    for i in range(2):
        record(f'low_map_G_{i+1}', 98, 'TCGAT', '5M', mq=5)
    for name, flag in [('duplicate_G', 1024), ('secondary_G', 256),
                       ('supplementary_G', 2048), ('qc_fail_G', 512)]:
        record(name, 98, 'TCGAT', '5M', flag=flag)

    record('insertion', 130, reference[130:134] + 'GG' + reference[134:138], '4M2I4M')
    record('deletion', 145, reference[145:149] + reference[151:155], '4M2D4M')
    record('soft_clip', 165, 'GG' + reference[165:171], '2S6M')
    record('ref_skip', 175, reference[175:178] + reference[182:185], '3M4N3M')
    left = record('pair_01', 20, reference[20:40], '20M', flag=99)
    right = record('pair_01', 60, reference[60:80], '20M', flag=147)
    left.next_reference_id = right.next_reference_id = 0
    left.next_reference_start, right.next_reference_start = 60, 20
    left.template_length, right.template_length = 60, -60
    record('unmapped', None, 'AACCGG', None, flag=4, mq=0)

    # Deliberately reverse record order: BAM encoding itself does not imply sorting.
    unsorted_path = work_dir / 'toy.unsorted.bam'
    with pysam.AlignmentFile(str(unsorted_path), 'wb', header=header) as bam:
        for read in reversed(records):
            bam.write(read)
    bam_path = work_dir / 'toy.sorted.bam'
    pysam.sort('-o', str(bam_path), str(unsorted_path))
    pysam.index(str(bam_path))
    return work_dir, fasta_path, bam_path


def keep_alignment(read, min_mapq=30):
    """Teaching policy; duplicate and other flags are supplied, not inferred here."""
    return (not read.is_unmapped and not read.is_secondary and not read.is_supplementary
            and not read.is_duplicate and not read.is_qcfail
            and read.mapping_quality != 255 and read.mapping_quality >= min_mapq)


def count_site(bam_path, position0, min_baseq=20, min_mapq=30, filter_flags=True):
    """Count bases at a zero-based coordinate with all teaching filters explicit."""
    counts = Counter()
    deletions = skips = 0
    with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
        if not 0 <= position0 < bam.get_reference_length('toy'):
            raise ValueError('Choose a zero-based position inside contig toy')
        # Disable implicit base/flag/BAQ rules, then apply our visible policy below.
        # All toy pairs are non-overlapping. The cap exceeds this dataset's depth.
        for column in bam.pileup('toy', position0, position0 + 1, truncate=True,
                                 stepper='nofilter', min_base_quality=0,
                                 min_mapping_quality=0, compute_baq=False,
                                 ignore_overlaps=False, ignore_orphans=False,
                                 max_depth=10000):
            for observation in column.pileups:
                read = observation.alignment
                if filter_flags:
                    if not keep_alignment(read, min_mapq):
                        continue
                elif read.mapping_quality == 255 or read.mapping_quality < min_mapq:
                    continue
                # N can also have is_del set. Test reference skips first.
                if observation.is_refskip:
                    skips += 1
                    continue
                if observation.is_del:
                    deletions += 1
                    continue
                query_index = observation.query_position
                if query_index is None or read.query_qualities is None:
                    continue
                if read.query_qualities[query_index] < min_baseq:
                    continue
                base = read.query_sequence[query_index]
                if base in 'ACGT':
                    counts[base] += 1
    depth = sum(counts.values())
    return {'counts': dict(counts), 'base_depth': depth,
            'g_fraction': counts['G'] / depth if depth else None,
            'deletion_observations': deletions, 'reference_skips': skips}


In [ ]:
work_dir, fasta_path, bam_path = create_bam_example()
print('Output directory:', work_dir)
for path in sorted(work_dir.iterdir()):
    print(path.name, path.stat().st_size, 'bytes')


The builder writes an intentionally unsorted BAM, then calls `pysam.sort` and
`pysam.index`. A `.bam.bai` is an index, not another set of reads. Sorting and
indexing change access to records; neither operation calls variants. A FASTA
index (`.fai`) is different from a BAM index.

The 200-base contig is named **toy**. At 1-based position 101, six good A and six
good G observations are supplemented by three low-base-Q G, two low-map-Q G,
and four G records with different flags. Other regions contain an insertion,
deletion, soft clip, reference skip, and one paired fragment. There is also an
unmapped record. These short reads illustrate fields, not realistic unique mappings.
Flags and mapping qualities are assigned for teaching, not inferred by an aligner.
Some secondary/supplementary records' primary counterparts are deliberately absent.


## 1. Read the header and count records

`AlignmentFile(..., 'rb')` opens compressed BAM for reading; `with` closes it
even if the cell raises an error. Inspect the reference names, lengths, and sort
order. `fetch(until_eof=True)` reads from the current file position through the
remaining records, including unmapped records. We reopen the file for each task
so a previous iterator cannot leave us at the end.

Predict whether the number of records equals the number of distinct query names.
Run the cell. Explain the difference; do not interpret either value as a general
count of independent molecules. Keeping a whole file in `all_records` is convenient
for 28 records; stream large BAM files instead.


In [ ]:
with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
    print('Reference names:', bam.references)
    print('Reference lengths:', bam.lengths)
    print('Sort order:', bam.header.to_dict()['HD']['SO'])
    print('Index available:', bam.has_index())
    all_records = list(bam.fetch(until_eof=True))
print('Alignment records:', len(all_records))
print('Mapped records:', sum(not r.is_unmapped for r in all_records))
print('Unmapped records:', sum(r.is_unmapped for r in all_records))
print('Unique query names:', len({r.query_name for r in all_records}))


**Your explanation:** Record your prediction, the changed output, and what it means.


## 2. Fetch a region: a row view of the alignments

With numeric arguments, `fetch('toy', start, stop)` uses **zero-based,
start-inclusive, stop-exclusive** coordinates. SAM text POS is 1-based:
`reference_start=98` is SAM POS=99. Query `[100,101)` for SAM position 101.
The index lets pysam retrieve overlapping alignments efficiently.

Predict the number of returned records. Run the cell, then try `[103,104)`.
Why does a read ending at 103 (excluded) not overlap a region starting at 103?
Restore `[100,101)` afterward. The printed SAM line is a readable representation
of a BAM record, not a view of raw binary bytes.


In [ ]:
region_start, region_stop = 100, 101  # Python: zero-based, excluded stop.
with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
    overlapping = list(bam.fetch('toy', region_start, region_stop))
print('Overlapping alignment records:', len(overlapping))
for read in overlapping[:4]:
    print(read.query_name, read.reference_start, read.reference_end,
          read.cigarstring, read.mapping_quality, read.query_sequence)
print('Selected record as a full SAM TEXT line:')
print(overlapping[0].to_string() if overlapping else '(empty region)')


**Your explanation:** Record your prediction, the changed output, and what it means.


## 3. Flags, mates, and stored orientation

Use the boolean properties rather than memorizing bit arithmetic. Flag categories
can overlap: a reverse-oriented mate is both paired and reverse, for example.
Find the two records for `pair_01`. Predict the mate coordinates and the sign of
their template lengths, then inspect the output.

Reverse-aligned SEQ is stored in alignment orientation. `get_forward_sequence()`
restores the original read orientation. Do not reverse-complement the stored
sequence again when extracting bases for a reference-coordinate pileup.


In [ ]:
flag_counts = {
    'reverse': sum(r.is_reverse for r in all_records),
    'paired': sum(r.is_paired for r in all_records),
    'duplicate': sum(r.is_duplicate for r in all_records),
    'secondary': sum(r.is_secondary for r in all_records),
    'supplementary': sum(r.is_supplementary for r in all_records),
    'QC failed': sum(r.is_qcfail for r in all_records),
}
print(flag_counts)
for read in all_records:
    if read.query_name == 'pair_01':
        print('pair_01:', 'read1=', read.is_read1, 'read2=', read.is_read2,
              'start=', read.reference_start, 'mate start=', read.next_reference_start,
              'template length=', read.template_length)
reverse_read = next(r for r in all_records if r.query_name == 'G_02')
print('Stored alignment-oriented sequence:', reverse_read.query_sequence)
print('Original read orientation:', reverse_read.get_forward_sequence())


**Your explanation:** Record your prediction, the changed output, and what it means.


## 4. CIGAR determines which coordinates a read consumes

Try `deletion`, `insertion`, `soft_clip`, and `ref_skip` in the cell. Before each
run, predict the stored query length and reference span. M consumes both sides
and allows a match or mismatch; I consumes only query; D and N consume only
reference; S consumes stored query but is not aligned to reference.

`get_aligned_pairs()` exposes query/reference index pairs. An insertion or soft
clip has no reference coordinate for those query bases. A deletion or reference
skip has no query base at those reference coordinates. **N is a reference skip,
not a synonym for a DNA deletion**; spliced RNA alignments commonly use N.
Our N record is an artificial API example, not structural-variant evidence.


In [ ]:
chosen_name = 'deletion'  # Also try insertion, soft_clip, and ref_skip.
chosen = next(r for r in all_records if r.query_name == chosen_name)
print('Name:', chosen.query_name, 'CIGAR:', chosen.cigarstring)
print('Stored query length:', chosen.query_length)
print('Reference interval:', (chosen.reference_start, chosen.reference_end))
print('Reference span:', chosen.reference_end - chosen.reference_start)
print('Pairs: (query index, reference index); None means no base on that side')
print(chosen.get_aligned_pairs())


**Your explanation:** Record your prediction, the changed output, and what it means.


## 5. Pileup: a column view, with explicit filtering

A region fetch returns records; `pileup` organizes them by reference position.
Our supplied `count_site` function disables the pileup engine's implicit filters
and then applies a visible teaching policy: exclude unmapped, secondary,
supplementary, duplicate, and QC-failed alignments; require mapping Q ≥ 30
and base Q ≥ 20. MAPQ 255 (unavailable) is excluded. These are example settings,
not a universal variant-calling recipe. Supplementary alignments may be useful
for SV analysis; their exclusion here is specific to this simple SNV count.

Predict the raw and filtered A/G counts. Run the cell. Lower **only** base Q to 0,
then restore 20 and lower **only** mapping Q to 0. Finally try base Q40.
Record how the numerator and denominator change, then restore defaults.
The helper has a depth cap of 10,000 (well above this dataset), disables BAQ,
and does not collapse overlapping mates; the single toy pair does not overlap.
Those choices must be reviewed for real datasets.


In [ ]:
position0 = 100  # SAM/VCF position 101.
min_baseq, min_mapq = 20, 30
raw_site = count_site(bam_path, position0, min_baseq=0, min_mapq=0, filter_flags=False)
filtered_site = count_site(bam_path, position0, min_baseq, min_mapq)
print('Raw observations:', raw_site)
print('After our filters:', filtered_site)


**Your explanation:** Record your prediction, the changed output, and what it means.


Inspect the short loop below at reference index 149. Then try 178 and 133.
Why must the code check `is_refskip` and `is_del` before reading `query_position`?
For the insertion, `indel=2` is reported at the preceding aligned base; inserted
bases do not occupy separate reference columns. A pileup column object is tied
to its iterator: copy values you need while iterating, rather than storing it
for later use after closing the file.


In [ ]:
# Inspect an insertion/deletion/skip without treating a missing query base as a letter.
inspect_position0 = 149  # Try 100, 133, or 178 as well.
with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
    for column in bam.pileup('toy', inspect_position0, inspect_position0 + 1,
                             truncate=True, stepper='nofilter', min_base_quality=0,
                             compute_baq=False, ignore_overlaps=False,
                             ignore_orphans=False, max_depth=10000):
        for observation in column.pileups:
            read = observation.alignment
            if not keep_alignment(read):
                continue
            if observation.is_refskip:
                label = 'reference skip (N): no query base'
            elif observation.is_del:
                label = 'deletion (D): no query base'
            else:
                label = read.query_sequence[observation.query_position]
            print(column.reference_pos, read.query_name, label,
                  'insertion/deletion length AFTER this base:', observation.indel)


**Your explanation:** Record your prediction, the changed output, and what it means.


## 6. Plot base coverage, including positions with no observations

`count_coverage` returns four arrays in A/C/G/T order. Summing them counts retained
bases, not merely alignments whose reference spans overlap a position. We pass
both a base-quality threshold and our read callback so the policy matches the
preceding analysis. The arrays include zeros for uncovered positions.

Predict the depth at index 100 and at deletion indices 149–150. Run the cell.
Explain why fetching a record overlapping index 149 does not give base coverage
of one there. The plot has a text fallback if matplotlib is unavailable.


In [ ]:
#@title Plot retained base coverage — supplied drawing code
plot_start, plot_stop = 90, 185
with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
    acgt = bam.count_coverage('toy', plot_start, plot_stop, quality_threshold=20,
                             read_callback=lambda read: keep_alignment(read, 30))
base_depth = [sum(values) for values in zip(*acgt)]
print('Every position is included, including zero-depth positions.')
for position in [90, 100, 130, 149, 150, 151, 178, 182]:
    print(position, base_depth[position - plot_start])
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print(list(zip(range(plot_start, plot_stop), base_depth)))
if plt is not None:
    figure, ax = plt.subplots(figsize=(10, 3.5))
    ax.step(range(plot_start, plot_stop), base_depth, where='mid', color='#0d7377')
    ax.set(xlabel='Reference position (0-based)', ylabel='Retained A/C/G/T observations',
           title='Synthetic BAM: base coverage, not alignment-span coverage', ylim=(0, 14))
    ax.axvspan(149, 151, color='#ad4936', alpha=.15, label='2D example: [149,151)')
    ax.legend()
    figure.tight_layout()
    plt.show()


**Your explanation:** Record your prediction, the changed output, and what it means.


## 7. Write and index a filtered regional BAM

Extract alignments overlapping `[90,110)` that pass the record-level flag and
mapping-quality policy. Reuse the input header with `template=source`. The
subset stays in coordinate order, so it can be indexed after the writer closes.

Predict how many records are written. Why will the output contain more records
than the default count of retained bases at position 101? This step applies no
per-base quality filter: low-quality bases remain attached to retained records.


In [ ]:
# This overwrites only our derived file in the fresh notebook output directory.
subset_path = work_dir / 'candidate.filtered.bam'
written = 0
with pysam.AlignmentFile(str(bam_path), 'rb') as source:
    with pysam.AlignmentFile(str(subset_path), 'wb', template=source) as output:
        for read in source.fetch('toy', 90, 110):
            if keep_alignment(read, 30):
                output.write(read)
                written += 1
# A subset selected in coordinate order remains coordinate-sorted.
# Close the writer BEFORE making an index or reopening the file.
pysam.index(str(subset_path))
with pysam.AlignmentFile(str(subset_path), 'rb') as subset:
    assert subset.has_index()
    print('Records written:', written, 'records fetched:', len(list(subset.fetch('toy', 100, 101))))
print('Subset evidence with base Q >= 20:', count_site(subset_path, 100))
print('BAM:', subset_path)
print('Index:', str(subset_path) + '.bai')


**Your explanation:** Record your prediction, the changed output, and what it means.


## Save your results

Save this notebook with outputs and explanations. The temporary directory contains
your synthetic BAM, its index, reference, and filtered subset. For region access,
keep a BAM and its `.bai` together. The next optional cell bundles the output
directory into a ZIP next to it. In Colab's Files panel, open the printed folder
path and download the ZIP; downloading the notebook alone does not include these files.


In [ ]:
import shutil
archive_path = Path(shutil.make_archive(str(work_dir) + '_results', 'zip', root_dir=work_dir))
print('Optional data download:', archive_path)


## Optional — Why region access needs an index

The cell makes a **copy** of the sorted BAM under a new name without copying its
index. A sequential scan still works, but a coordinate query should fail. Run it,
then explain which operation requires indexed access. The original BAM is intact.


In [ ]:
no_index_path = work_dir / 'copy_without_index.bam'
shutil.copyfile(bam_path, no_index_path)
with pysam.AlignmentFile(str(no_index_path), 'rb', index_filename=None) as bam:
    print('Index available:', bam.has_index())
    print('Sequential records:', sum(1 for _ in bam.fetch(until_eof=True)))
with pysam.AlignmentFile(str(no_index_path), 'rb') as bam:
    try:
        list(bam.fetch('toy', 100, 101))
    except ValueError as error:
        print('Expected regional-query failure:', error)


**Your explanation:** Record your prediction, the changed output, and what it means.


## Sources

- [pysam: working with BAM files](https://pysam.readthedocs.io/en/stable/usage.html)
- [pysam API: AlignmentFile, fetch, pileup, and count_coverage](https://pysam.readthedocs.io/en/stable/api.html)
- [pysam installation](https://pysam.readthedocs.io/en/stable/installation.html)
- [SAM/BAM specification](https://samtools.github.io/hts-specs/SAMv1.pdf)

Synthetic records and exercises were authored for this course. Notebook form
packaging uses the shared helper and its retained upstream Apache license.
